In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Function for plotting predicted values against actual values for each lattice parameter (a, b, and c)
def PlotPredictions(Y_test, Y_pred_df, modelName, displayStatistics=True, MSEVals=None, MAEVals=None, R2Vals=None, df=None, col=None, labels=None, displayOthers=True):
    fig, axes = plt.subplots(1, 3, figsize=(18, 10))

    Y_test_df = Y_test.reset_index(drop=True)

    # Creates 3 subplots for each lattice parameter
    for i, param in enumerate(Y_test_df.columns):
        ax = axes[i]

        #This plots each point colored based off its chosen label if X_test_full and labels is passed
        if col is not None and labels is not None:            
            X_test_full = df.loc[Y_test.index].reset_index(drop=True)
            
            arr = np.asarray(X_test_full[col])
            missing = [f for f in labels if f not in arr]
            if missing:
                raise ValueError(f"The following labels are not found in column {col} of X_test_full: {missing}")

            labelsList = list(labels)
            cmap = plt.get_cmap('tab10')
            colors = {label: cmap(i % cmap.N) for i, label in enumerate(labelsList)}
            
            labelsList = ["Other"] + labelsList
            colors["Other"] = "gray"
            
            for label in labelsList:
                if label == 'Other':
                    mask = ~np.isin(X_test_full[col], labels)
                    if not displayOthers or not mask.any():
                        continue
                else:      
                    mask = X_test_full[col] == label
                    
                # Plot each actual value against its predicted value as individual points
                ax.scatter(
                    Y_test_df.loc[mask, param],
                    Y_pred_df.loc[mask, f"{param}_pred"],
                    color = colors[label],
                    label = label.title() if mask is not None else None,
                    alpha = 0.7
                )

                # Create line of perfect fit where actual value = predicted value
                line = [Y_test_df[param].min(), Y_test_df[param].max()]
                ax.plot(line, line, 'k--')
            
            if col == 'crystal_system':
                ax.legend(title='Crystal System', fontsize=18, title_fontsize=18)
            else:
                ax.legend(title=col, fontsize=18, title_fontsize=18)
        
        #This plots each point as the same color
        else:
            # Plot each actual value against its predicted value as individual points
            ax.scatter(Y_test_df[param], Y_pred_df[f"{param}_pred"])

            # Create line of perfect fit where actual value = predicted value
            line = [Y_test_df[param].min(), Y_test_df[param].max()]
            ax.plot(line, line, 'r--')

        ax.set_xlabel("Actual Value", fontsize=18)
        ax.tick_params(labelsize=18)
        ax.set_ylabel("Predicted Value", fontsize=18)
        ax.set_title(f"Lattice Parameter '{param}' (Å)", fontsize=20)
        
        # Display MSE and R2 statistics for each lattice parameter if desired
        if displayStatistics:
            if MSEVals is not None and MAEVals is not None and R2Vals is not None:
                if df is not None:
                    text = f"MSE={MSEVals[i]:.4f}\nMAE={MAEVals[i]:.4f}\nR²={R2Vals[i]:.4f}\nTest Dataset Size={len(df)}"
                else:
                    text = f"MSE={MSEVals[i]:.4f}\nMAE={MAEVals[i]:.4f}\nR²={R2Vals[i]:.4f}"
                
                ax.text(
                    0.05, 0.95,
                    text,
                    transform=ax.transAxes,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor="gray"),
                    fontsize=14
                )
    
    fig.suptitle(f"Predicted vs. Actual Lattice Parameter Using {modelName}", fontsize=22)
    
    plt.tight_layout()
    plt.show()

# Initialize dataset paths (Experimantlly Observed)
# MPDatasetFeaturized = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Featurized_Experimental.csv"
# TrainingDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/Training_Dataset_Experimental_CrossVal.csv"
# ModelMetricsDir = "<local_path>/FuelComp/MachineLearning/Dataset/ModelMetrics_Experimental_CrossVal.csv"
# PredictionDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/DatasetPredictions_Experimental_CrossVal.csv"

# # Initialize dataset paths (Theoretical Included)
MPDatasetFeaturized = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Featurized_Full.csv"
TrainingDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/Training_Dataset_Full_AvgOnly_CrossVal.csv"
ModelMetricsDir = "<local_path>/FuelComp/MachineLearning/Dataset/ModelMetrics_Full_AvgOnly_CrossVal.csv"
PredictionDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/DatasetPredictions_Full_AvgOnly_CrossVal.csv"

In [ ]:
# Read in featurized dataset and feature labels

df = pd.read_csv(TrainingDatasetDir, low_memory=False)
display(df.head())
display(len(df))

featureLabels = joblib.load('ML_FeatureLabels_Full_AvgOnly.joblib')
print(featureLabels)

In [ ]:
elementCols = ['H', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu']

physFeatures = [feat for feat in featureLabels if feat not in elementCols]

print(physFeatures)

In [ ]:
df_cubic = df[df['crystal_system'] == 'cubic']

correlations = df_cubic[['a'] + featureLabels].corr(method='spearman')

a_corr = correlations['a'].drop('a')

corr_df = a_corr.reset_index()
corr_df.columns = ['Feature', 'Correlation']

corr_df['AbsCorr'] = corr_df['Correlation'].abs()
corr_df_sorted = corr_df.sort_values(by='AbsCorr', ascending=False)

filtered_corr = corr_df_sorted[corr_df_sorted['AbsCorr'] > 0.2]

display(filtered_corr)

In [ ]:
physFeats = [feat for feat in filtered_corr['Feature'] if feat in physFeatures]
elementFeats = [el for el in elementCols if el in df_cubic.columns]

print(physFeats)

print("Number of physical features in symbolic expression:", len(physFeats))
print("Number of elements in symbolic expression:", len(elementFeats))

In [ ]:
targetCol = 'a'

allFeats = elementFeats + physFeats + ['spacegroup_num']
print("Total number of features in symbolic expression:", len(allFeats))

df_model = df_cubic[[targetCol] + allFeats].dropna()
X = df_model[allFeats]
Y = df_model[targetCol]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, Y_train)
Y_pred = model.predict(X_test)
mae = mean_absolute_error(Y_test, Y_pred)

# Plot predictions vs true values
plt.figure(figsize=(6, 6))
plt.scatter(Y_test, Y_pred, alpha=0.6, edgecolor='k')
plt.plot([Y.min(), Y.max()], [Y.min(), Y.max()], 'r--', label='Ideal Fit')
plt.xlabel('Actual Lattice Parameter (Å)')
plt.ylabel('Predicted Lattice Parameter (Å)')
plt.title(f'Symbolic Model Prediction for Cubic Crystal Systems\nMAE = {mae:.3f} Å')
plt.legend()
plt.tight_layout()
plt.show()

best_eq_str = f"Lattice Parameter = {model.intercept_:.4f} " + " ".join([f"+ {coef:.4f}*({feat})" if coef >= 0 else f"- {abs(coef):.4f}*({feat})"for coef, feat in zip(model.coef_, allFeats)])

display(best_eq_str)
